# 一、核心定义
- Python 的魔法属性（Magic Attributes） 也叫特殊属性 / 方法（Special Methods），是指以双下划线 __ 开头和结尾（__xxx__）的内置属性 / 方法。它们不是由开发者手动调用，而是在特定场景下由 Python 解释器自动触发，用于定制类的核心行为（如初始化、字符串表示、容器操作、运算符重载等）。
- 魔法属性是 Python 面向对象的核心特性，让自定义类可以模拟列表、字典等内置类型的行为，大幅提升代码的灵活性和可读性。
# 二、常见魔法属性
## （一）构造/初始化相关
1. \_\_init__：实例初始化方法
    - 触发场景：实例化类时自动个调用，在\_\_new__之后。
    - 核心作用：初始化实例属性，是最常用的魔法方法。
    - 案例：

In [1]:
class Person:
    # 初始化实例属性：姓名、年龄
    def __init__(self, name, age):
        self.name = name
        self.age = age

# 实例化时自动调用__init__
p = Person("张三", 20)
print(p.name, p.age)  # 输出：张三 20

张三 20


2. \_\_new__：实例创建方法
    - 触发场景：实例化类时最先执行（比\_\_init__早），是类的“构造方法”。
    - 核心作用：创建并返回实例对象，常用于单例模式、不可变类型定制。
    - 案例：

In [2]:
class Singleton:
    _instance = None  # 存储唯一实例

    def __new__(cls):
        # 若实例不存在，创建；否则返回已有实例
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

s1 = Singleton()
s2 = Singleton()
print(s1 is s2)  # 输出：True（全局唯一实例）

True


## （二）字符串表示相关
1. \_\_str__：友好字符串表示
    - 触发场景：被print()、str()触发
    - 核心作用：返回人类易读的字符串，面向“用户”
    - 案例：

In [3]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"【用户】{self.name}，年龄{self.age}"

p = Person("李四", 25)
print(p)  # 输出：【用户】李四，年龄25
print(str(p))  # 输出同上

【用户】李四，年龄25
【用户】李四，年龄25


2. \_\_repr__：官方字符串表示
    - 触发场景：被repr()、交互式解释器触发；若未定义\_\_str__，print()会fallback到\_\_repr__。
    - 核心作用：返回可重建实例的字符串（尽量满足eval(repr(obj)) == obj），面向“开发者”。
    - 案例：

In [4]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __repr__(self):
        # 格式：类名(参数)，可直接用于创建实例
        return f"Person('{self.name}', {self.age})"

p = Person("王五", 30)
print(repr(p))  # 输出：Person('王五', 30)
# 交互式环境中直接输入p，会显示该结果

Person('王五', 30)


## （三）容器/序列相关（模拟列表/字典）
让自定义类支持len()、索引、切片等容器操作。
1. \_\_len__：返回长度
    - 触发场景：被len()触发。
    - 核心作用：定义实例的“长度”，需返回整数。
    - 案例：

In [5]:
class StudentList:
    def __init__(self):
        self.students = []

    def add(self, name):
        self.students.append(name)

    def __len__(self):
        return len(self.students)

sl = StudentList()
sl.add("小明")
sl.add("小红")
print(len(sl))  # 输出：2

2


2. \_\_getitem__：获取元素
    - 触发场景：被obj[key]、切片(obj[1:])触发。
    - 核心作用：支持索引/键访问元素。
    - 案例：

In [6]:
class StudentList:
    def __init__(self):
        self.students = ["小明", "小红", "小刚"]

    def __getitem__(self, index):
        return self.students[index]

sl = StudentList()
print(sl[0])     # 输出：小明
print(sl[1:])    # 输出：['小红', '小刚']（支持切片）

小明
['小红', '小刚']


3. \_\_setitem__：设置元素
    - 触发场景：被obj[key] = value触发。
    - 核心作用：支持修改容器元素。
    - 案例：

In [7]:
class StudentList:
    def __init__(self):
        self.students = ["小明", "小红", "小刚"]

    def __setitem__(self, index, value):
        self.students[index] = value

sl = StudentList()
sl[0] = "小李"
print(sl.students)  # 输出：['小李', '小红', '小刚']

['小李', '小红', '小刚']


4. \_\_delitem__：删除元素
    - 触发场景：被del obj[key]触发。
    - 核心作用：支持删除容器元素。
    - 案例：

In [8]:
class StudentList:
    def __init__(self):
        self.students = ["小明", "小红", "小刚"]

    def __delitem__(self, index):
        del self.students[index]

sl = StudentList()
del sl[1]
print(sl.students)  # 输出：['小明', '小刚']

['小明', '小刚']


## （四）可调用相关（实例模拟函数）
1. \_\_call__：实例可调用
    - 触发场景：被obj()触发，实例像函数一样调用。
    - 核心作用：实现“可调用对象”，常用语装饰器、工厂模式。
    - 案例：

In [9]:
class Calculator:
    def __call__(self, a, b):
        return a + b

calc = Calculator()
print(calc(10, 20))  # 输出：30（实例像函数一样调用）

30


## （五）元信息/文档信息
1. \_\_dict__：属性字典
    - 触发场景：直接访问obj.\_\_dict__。
    - 核心作用：存储实例的属性和值（可直接修改）。
    - 案例：

In [10]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("赵六", 35)
print(p.__dict__)  # 输出：{'name': '赵六', 'age': 35}
# 修改__dict__直接改变实例属性
p.__dict__["age"] = 36
print(p.age)  # 输出：36

{'name': '赵六', 'age': 35}
36


2. \_\_class__：所属类
    - 触发场景：访问obj.\_\_class__。
    - 核心作用：返回实例所属的类，可用于动态创建实例。
    - 案例：

In [11]:
p = Person("赵六", 35)
print(p.__class__)  # 输出：<class '__main__.Person'>
# 通过__class__创建新实例
p2 = p.__class__("孙七", 40)
print(p2.name)  # 输出：孙七

<class '__main__.Person'>
孙七


3. \_\_doc__：文档字符串
    - 触发场景：访问Class.\_\_doc__或method.\_\_doc__。
    - 核心作用：返回类 / 方法的注释（docstring），用于生成文档。
    - 案例：

In [12]:
class Person:
    """这是一个人类的类，用于存储姓名和年龄"""
    def __init__(self, name, age):
        """初始化方法：
        参数：
            name: 姓名（字符串）
            age: 年龄（整数）
        """
        self.name = name
        self.age = age

print(Person.__doc__)  # 输出：这是一个人类的类，用于存储姓名和年龄
print(Person.__init__.__doc__)  # 输出初始化方法的注释

这是一个人类的类，用于存储姓名和年龄
初始化方法：
参数：
    name: 姓名（字符串）
    age: 年龄（整数）



## （六）运算重载相关
1. \_\_add__：重载加法运算符
    - 触发场景：被 obj1 + obj2 触发。
    - 核心作用：自定义 “+” 的行为。
    - 案例：

In [13]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        # 自定义点的加法：x/y分别相加
        return Point(self.x + other.x, self.y + other.y)

    def __str__(self):
        return f"Point({self.x}, {self.y})"

p1 = Point(1, 2)
p2 = Point(3, 4)
p3 = p1 + p2
print(p3)  # 输出：Point(4, 6)

Point(4, 6)


2. \_\_eq__：重载相等运算符
    - 触发场景：被 obj1 == obj2 触发（默认比较内存地址）。
    - 核心作用：自定义 “相等” 的判断规则。
    - 案例：

In [14]:
class Person:
    def __init__(self, id, name):
        self.id = id  # 唯一标识
        self.name = name

    def __eq__(self, other):
        # 自定义规则：id相同则相等
        return self.id == other.id

p1 = Person(1, "张三")
p2 = Person(1, "张三")
p3 = Person(2, "李四")
print(p1 == p2)  # 输出：True
print(p1 == p3)  # 输出：False

True
False


## （七）迭代相关
1. \_\_iter__ + \_\_next__：支持迭代
    - 触发场景：for 循环、iter() 触发 \_\_iter__；next() 触发 \_\_next__。
    - 核心作用：让自定义类支持迭代（像列表一样用 for 循环）。
    - 案例（模拟 range）：

In [15]:
class MyRange:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __iter__(self):
        return self  # 返回迭代器自身

    def __next__(self):
        if self.start >= self.end:
            raise StopIteration  # 终止迭代
        current = self.start
        self.start += 1
        return current

# 模拟 range(1, 4)
for i in MyRange(1, 4):
    print(i)  # 输出：1 2 3

1
2
3
